# Step 21.1 — Generation loop：一个预测怎样成为下一轮输入

先在仓库根目录同步母版：

```bash
git pull
```

再创建独立工作副本；仓库已有 `notebooks/work/`，时间戳与 `cp -n` 避免覆盖笔记：

```bash
cp -n notebooks/92_step21_1_generation_loop.ipynb notebooks/work/92_step21_1_generation_loop_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

命令是给以后使用的参考，本次没有执行它们。请只在新副本中添加自己的笔记或练习。MLX 固定 **v0.32.2**；全部代码单元未执行、输出为空。

本节只建立一条非空 prompt 的固定长度生成循环：**context → 最后位置 logits → 下一个 token → append → repeat**。先读手算与单步，再读完整函数。Greedy 在这里仅作最小选择规则，21.2 才比较选择策略；本节没有 EOS 停止、batch 生成、context cropping 或 KV cache。

2026-11-17 教学修订 **1/2**：将原来集中交付的 helper 与循环拆成解释和小代码，保留原有函数签名、检查规则与生成行为。历史初稿为 0/2；其他小节的历史计数不变。没有读取语料、加载权重、运行模型或生成真实结果。

## 1. 从已学内容接到生成：变的是输入序列

[6.3 Next-token shift](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c707b666b91e536f599437a4cab85d9fa1cd6485/notebooks/19_step06_3_next_token_shift.ipynb) 已经说明：输入位置 `t` 的输出用于预测它右边的 token；[15.4 MiniGPT](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c707b666b91e536f599437a4cab85d9fa1cd6485/notebooks/65_step15_4_implement_minigpt.ipynb) 将整数 IDs `(B,T)` 映射为 raw logits `(B,T,V)`；[17.1 交叉熵](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c707b666b91e536f599437a4cab85d9fa1cd6485/notebooks/72_step17_1_logits_probabilities_cross_entropy.ipynb) 再把这些分数与已知 target 比较。

训练有真实文本作答案。例如输入 `A B`，相应 target 可以是 `B C`。模型在两个位置分别答题，loss 用答案评分。生成时只有 prompt（提示前缀）`A B`，后面的答案尚不存在。因此不能先准备全部未来输入，也不能把每个位置的预测一起当作续文。

| 位置 | 该位置看见的前缀 | 该位置的 logits 回答什么 |
|---|---|---|
| 0 | `A` | `A` 后面是谁？ |
| 1 | `A B` | `A B` 后面是谁？——当前待续写的位置 |

我们只取最后一行，选一个 token，接在 `A B` 后面，再让模型处理这个新前缀。这叫 **autoregressive generation（自回归生成）**：下一轮条件包含自己刚选出的结果。这里追加的是整数 token ID，不是 logits、概率或 embedding 向量。

生成期间没有 target、loss、反向传播或 Adam 更新。模型参数保持原值，循环不断增长的是输入 IDs。Step 20 学的是保存与恢复这些参数和相关状态；本节不会重新训练模型，也不会自动加载任何 checkpoint。

## 2. 先约定输入、输出与每一条轴

本节令 `B=1`，`L` 为原 prompt 长度，`r` 为已经追加的 token 数，`V` 为词表大小。第 `r` 轮开始时序列长度为 `L+r`。

| 对象 | Shape | 每个数的含义 |
|---|---|---|
| `ids` | `(1,L+r)` | 已有 token 的整数 ID |
| `logits_fn(ids)` | `(1,L+r,V)` | 每个位置对 V 个下一 token 候选的分数 |
| 最后位置 `last` | `(1,V)` | 当前前缀之后的候选分数 |
| 选中的 `token_id` | `(1,)` | 一条序列选出一个整数 ID |
| 增轴后的 `nxt` | `(1,1)` | 可以沿时间轴拼接的一列 ID |
| 拼接后的 `ids` | `(1,L+r+1)` | 原前缀加一个新 token |

`logits_fn` 是 **callback（回调函数）**：把“怎样产生分数”作为函数参数交给循环。Python 中 `toy_logits` 表示函数本身，`toy_logits(ids)` 才调用它。循环只约定输入输出形状，不负责实现 Transformer。以后同一契约可以由 MiniGPT 实例提供；现在用一张人工表，隔离出序列管理问题。

若用概率语言描述模型，在固定参数 $\theta$ 下：

$$
p_\theta(y_1,\ldots,y_n\mid\text{prompt})
=\prod_{r=0}^{n-1}p_\theta(y_{r+1}\mid\text{prompt},y_1,\ldots,y_r).
$$

乘积表示先预测第一项，再在它已给定的条件下预测第二项，依次继续。循环实现的是这个条件更新顺序。本节的 greedy 规则只取每一步最大分数，并不保证最终整串序列的联合概率最大；这与逐步按分布抽样也是不同策略。

## 3. 在写循环前，手算三次状态变化

固定小词表：`A→0, B→1, C→2, _→3`。下面每一行都是**人为给定**的下一 token 分数，列按原 token ID 排列：

| 当前 token | A / ID 0 | B / ID 1 | C / ID 2 | _ / ID 3 |
|---|---:|---:|---:|---:|
| A | 0 | 3 | 1 | -1 |
| B | 0 | 0 | 3 | -1 |
| C | 3 | 0 | 0 | -1 |
| _ | 3 | 1 | 0 | -1 |

分数 `3` 不是 token ID，也不是概率。A 行的最大值是 3，它所在的**列索引 1**才是选择出的 ID，即 B。这个 toy 只依赖当前 token，故意没有模拟 Transformer 对整个前缀的注意力。

从 prompt `[A]` 开始，每轮只追加一个 token：

| 轮次 r | 调用前的 IDs | callback shape | 取出的最后一行 | 选中 ID | 追加后的 IDs |
|---:|---|---|---|---:|---|
| 0 | `[0]` | `(1,1,4)` | A 行 `[0,3,1,-1]` | 1 | `[0,1]` |
| 1 | `[0,1]` | `(1,2,4)` | B 行 `[0,0,3,-1]` | 2 | `[0,1,2]` |
| 2 | `[0,1,2]` | `(1,3,4)` | C 行 `[3,0,0,-1]` | 0 | `[0,1,2,0]` |

第二轮 callback 返回两行：第一行仍回答“A 后面是谁”，第二行才回答当前续写位置。我们不拿第一行覆盖已有 B。第三轮同理，只在末尾追加 A。

表中的 `ABCA` 是从指定分数表推出来的数学结果，**不是本次程序输出**，也不是模型学会语言的证据。下面的小代码逐步对应这张表。

## 4. 先建立人工回调，不复制整套 MiniGPT

`TRANSITION` 是 `(4,4)` 数组，行索引是当前 token ID，列索引是下一 token 候选 ID。数组索引 `TRANSITION[ids]` 为 `ids` 的每个元素取一整行：例如 `ids=[[0,1]]`，取 A 行和 B 行，结果 shape 是 `(1,2,4)`。

这与 embedding 查表的索引方式相似，但表里的每行是 **logits**，不是学习得到的特征向量。ID 必须在 `0..3` 内。`dtype=mx.float32` 指定分数类型，后面的输入 IDs 使用 `mx.int32`。

这两个单元是本节完整的 toy 依赖，之后的示例只用它们，不需要运行旧 Notebook 或借用旧 kernel 变量。

In [ ]:
import math
import mlx.core as mx

# 人工分数表；不是训练结果。此单元未执行。
TOY_VOCAB = ("A", "B", "C", "_")
TRANSITION = mx.array([[0., 3., 1., -1.],
                       [0., 0., 3., -1.],
                       [3., 0., 0., -1.],
                       [3., 1., 0., -1.]], dtype=mx.float32)

In [ ]:
def toy_logits(ids):
    # 输入 int32 (B,T)，合法 ID 0..3；输出 (B,T,4)。
    # 这个模型只看当前 token，故意不模拟真实 Transformer。
    return TRANSITION[ids]

## 5. 把第一轮摊开：四个动作分别做什么？

### 5.1 Feed context：给回调当前整条前缀

`[0]` 是一条 Python ID 列表；外面再套一层得到 `[[0]]`，表示一个 batch 中只有一条序列。`walk_ids` 是单步讲解用的名字，与后面的完整函数内部变量分开。

下面先得到整个 `(B,T,V)` 输出，还没有挑 token。`walk_logits` 是分数数组对象，得到这个对象不等于已经显示或测量过数值，沿用 Step 18 的惰性求值含义。

In [ ]:
walk_ids = mx.array([[0]], dtype=mx.int32)  # (1,1): prompt A
walk_logits = toy_logits(walk_ids)          # (1,1,4)

### 5.2 Take last logits：只取时间轴最后位置

`[:, -1, :]` 的三个部分分别对应 batch、time、vocabulary：第一个 `:` 保留所有 batch 行，`-1` 选时间轴最后一项，最后的 `:` 保留全部候选。用整数索引选定 time 后，这条轴消失，所以 `(1,T,4)` 变为 `(1,4)`。

`[:, :, -1]` 则会取**最后一个词表候选**在所有时间位置的分数，shape 为 `(1,T)`，不能拿它选择续文。取最后位置也不等于删掉输入前缀；我们只缩小用于本轮选择的输出。

In [ ]:
walk_last = walk_logits[:, -1, :]  # (1,4): A 行的全部候选分数

### 5.3 Choose：取最大值的位置，再恢复长度为 1 的时间轴

**Argmax（最大值索引）**返回最大分数所在的位置。对 `[0,3,1,-1]`，最大分数为 3，索引为 1；我们需要的是索引 1。`axis=-1` 明确沿词表轴选择，不能省略后把整个 batch 展平选择。

固定 v0.32.2 的 `mx.argmax` 默认 `keepdims=False`，会移除所选轴，输出为 `uint32`；这里用 `.astype(mx.int32)` 与输入 IDs 对齐。得到 `(1,)` 后，`[:, None]` 插入一个新轴，得到 `(1,1)`。它不添加新 token，只改变形状。

为什么没有先 softmax？对有限分数，指数函数单调、各项共享同一个正分母，所以最大分数与最大概率的位置相同。当前例子最大值唯一；并列分数的选择约定和 sampling 留到 21.2，不在这里把并列处理宣称为语言偏好。

In [ ]:
walk_token_id = mx.argmax(walk_last, axis=-1).astype(mx.int32)  # (1,)
walk_next = walk_token_id[:, None]                              # (1,1)

### 5.4 Append：沿时间轴连接，并把新序列交给下一轮

`mx.concatenate([walk_ids, walk_next], axis=1)` 将两个二维数组沿 time 轴连接。除了连接轴，其他维度必须一致；本例都是 batch 长度 1。结果从 `(1,1)` 变成 `(1,2)`，内容按手算为 `[[0,1]]`。

`walk_ids = ...` 是把名字绑定到新结果，不会把原 prompt 的 token 改成预测 token。若沿 `axis=0` 拼接，就成了另一条 batch 行，不是续写。`mx.stack` 会新增轴，也不等同于这里沿已有轴延长。

最后的 `mx.eval(walk_ids)` 明确完成当前序列及其依赖，随后才从这一状态继续。它不更新参数，也不是 `model.eval()`。这条逐轮边界用于看清流程，不是最快生成实现的承诺。

In [ ]:
walk_ids = mx.concatenate([walk_ids, walk_next], axis=1)  # (1,2)
mx.eval(walk_ids)
# 第一轮结束；没有打印值。下一轮应把新的 walk_ids 再传给 toy_logits。

## 6. 从一轮到 n 轮：循环不变量与容量

**Loop invariant（循环不变量）**是每轮都成立的关系：在第 `r` 轮开始，IDs 的前 L 项仍是原 prompt，末尾恰有 r 个已生成 token，shape 为 `(1,L+r)`。一轮完成一次 callback、选择和追加，于是下一轮长度是 `L+r+1`；没有任何一步覆盖前 L 项。

`range(n)` 给出 `0..n-1`，所以循环恰好追加 n 项；返回长度为 `L+n`。当 `n=0` 时没有 callback 或 forward，只把合法 prompt 转为 `(1,L)` 数组返回。

本节不裁剪上下文。若 `n>0`，各次 forward 的输入长度为：

$$
L,\ L+1,\ \ldots,\ L+n-1.
$$

因此要求最后一次输入也满足 `L+n-1 <= block_size`。例如 L=2、n=3，三次 forward 长度是 2、3、4：容量 4 足够，返回长度却是 5，因为最后追加的 token 尚未再次送入模型。若再多生成一项，就需要处理长度 5，应在开始之前拒绝，不能算到半途才报错。

`block_size` 是此函数允许的输入容量，不是自动裁剪开关。以后接真实 MiniGPT 时还须保证它不超过 `model.max_seq_len` 和约定的有效上下文范围。21.5 才讨论将完整 history 与模型窗口分开。

## 7. 为完整函数补上两道接口检查

前面的单步输入由本节手工指定；完整函数接收调用者输入，需要明确失败条件。

`check_prompt` 先确认非空 list/tuple，再检查每一项是 Python 整数且落在 `[0,V)`。`any(...)` 表示只要有一个无效项就拒绝；`type(t) is int` 也排除把布尔值当 ID。空 prompt 没有“最后位置”，不能继续。它只检查输入，不调用模型。

In [ ]:
def check_prompt(prompt, vocab_size):
    if not isinstance(prompt, (list, tuple)) or not prompt:
        raise ValueError("prompt must be a nonempty list/tuple of IDs")
    if any(type(t) is not int or not 0 <= t < vocab_size for t in prompt):
        raise ValueError("prompt contains an invalid token ID")

`last_logits` 把“调用回调 → 检查 shape → 取末行”集中为一个 helper。`(*ids.shape, vocab_size)` 是 tuple 展开：当 `ids.shape=(1,T)` 时它就是 `(1,T,V)`。如果 callback 已经只返回 `(1,V)`，或词表轴大小不符，会在选择前报错。

末行转为 float32 后，`mx.isfinite` 为每个分数产生布尔值；`mx.all` 把“全部有限”归约为标量，`.item()` 取回 Python 值供 `if` 判断。NaN/Inf 不应被悄悄当作正常分数。这里按原接口只检查**用于选择的末行**，不是对整个模型数值健康的证明，也不是后续 top-k 的屏蔽接口。

检查与取值包含 host 操作；这仍是普通 Python 教学 helper，不交给 `mx.compile`。求值边界沿用 [18.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c707b666b91e536f599437a4cab85d9fa1cd6485/notebooks/78_step18_2_eval_boundary.ipynb)，不是新训练算法。

In [ ]:
def last_logits(logits_fn, ids, vocab_size):
    logits = logits_fn(ids)
    if logits.shape != (*ids.shape, vocab_size):
        raise ValueError("callback must return (B,T,V) logits")
    last = logits[:, -1, :].astype(mx.float32)
    mx.eval(last)
    if not bool(mx.all(mx.isfinite(last)).item()):
        raise ValueError("model logits must be finite")
    return last

## 8. 完整函数：整合上面已拆解的步骤

这是前面单步、检查和容量规则的**汇总代码**，没有隐藏的新生成策略。首次阅读可以按下表逐段对照；理解流程后把它作为调用接口使用，不需要再抄一份相同函数。

| 函数中的部分 | 已解释的位置 | 为什么放在这里 |
|---|---|---|
| prompt、n、容量检查 | 第 6–7 节 | 在任何 forward 之前拒绝不合法请求 |
| 创建 `ids` | 第 5.1 节 | 把一条 prompt 放入 batch 轴 |
| `last_logits` | 第 5.2、7 节 | 回调输出与词表/时间轴契约一致 |
| `argmax` 与增轴 | 第 5.3 节 | 从分数选整数 ID，并准备拼接 |
| concatenate 与 eval | 第 5.4 节 | 保留前缀，完成新的当前序列 |
| `return ids` | 第 6 节 | 返回 prompt 加 n 个新 token |

签名中 `*` 后的 `vocab_size`、`block_size` 必须按名字传入，避免把词表大小与上下文容量混淆。`for _ in range(...)` 中 `_` 表示循环序号没有参与计算，当前状态由 `ids` 保存。

In [ ]:
def generate_greedy(logits_fn, prompt, max_new_tokens, *, vocab_size, block_size):
    check_prompt(prompt, vocab_size)
    if type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError("max_new_tokens must be a nonnegative integer")
    if type(block_size) is not int or block_size <= 0:
        raise ValueError("block_size must be a positive integer")
    # 最后一次 forward 的长度是 L+n-1；n=0 不调用模型。
    if max_new_tokens and len(prompt) + max_new_tokens - 1 > block_size:
        raise ValueError("context too long; cropping is introduced in 21.5")
    ids = mx.array([prompt], dtype=mx.int32)
    for _ in range(max_new_tokens):
        last = last_logits(logits_fn, ids, vocab_size)
        nxt = mx.argmax(last, axis=-1).astype(mx.int32)[:, None]
        ids = mx.concatenate([ids, nxt], axis=1)
        mx.eval(ids)  # 清晰的逐步求值边界，不是训练或反向传播。
    return ids

## 9. 可选的完整调用：先预测，再由自己决定是否运行

下面沿用第 3 节同一个 prompt、同一分数表和三轮过程，供以后工作副本使用。不是新的大实验，也不读取真实模型。先用表格预测返回 shape 与每个 token，再看代码如何显示它们。

`result.tolist()` 将数组值转为嵌套 Python 列表，`[0]` 取唯一 batch 行；`TOY_VOCAB[t]` 将 ID 查回字符，`"".join(...)` 把字符连接成字符串。没有 tokenizer 重新编码或隐藏采样。这些显示操作在未来执行时会读到数值；当前没有输出记录。

In [ ]:
# 可选：未来工作副本中的完整调用；本次未执行。
result = generate_greedy(toy_logits, [0], 3, vocab_size=4, block_size=4)
mx.eval(result)
print(result.shape, result.tolist())
print("".join(TOY_VOCAB[t] for t in result.tolist()[0]))

## 10. 将来接真实模型时，哪些可以复用？

可以复用的是 `generate_greedy` 的序列管理和 `(B,T) → (B,T,V)` 契约。人工 `TRANSITION` 不能带入真实模型；它的四字符映射也不能拿来解释另一个 checkpoint。

已有真实产物时，先按 [20.4 恢复流程](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c707b666b91e536f599437a4cab85d9fa1cd6485/notebooks/90_step20_4_resume_training.ipynb) 与 [20.5 核对材料](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c707b666b91e536f599437a4cab85d9fa1cd6485/notebooks/91_step20_5_resume_verification.ipynb) 取得匹配的架构、权重和完整字符映射，确认合法 ID、V 与位置容量，再把那个已恢复的模型实例作为 `logits_fn`。这是未来接入条件，本节不创建另一套模型来覆盖恢复结果，也不自动执行任何恢复步骤。

`model.eval()` 切换模型模式；若有 dropout 等依赖模式的行为，推理时需要相应模式，使用后恢复原模式。它不负责完成数组计算；`mx.eval(...)` 才是这里讨论的求值请求。当前 toy 没有模型模式。

当前函数每轮重新处理整个前缀，没有 KV cache；固定次数结束，没有 EOS；只从一条 prompt 创建 `B=1`，不是批量生成器。读到这些边界，应知道何时本函数不适用，不应通过扩大 `block_size` 冒充长上下文能力。

下一节 **21.2 Greedy vs sampling** 只替换“怎样从末行 logits 选择下一个 ID”这一环，并解释随机 key。循环管理前缀的职责保持不变。

## 11. 自己检查理解（不附答案）

1. 手写 prompt=[B,C] 的前两轮输入、最后 logits 与输出。
2. 解释为什么不能把 `logits[:, :, -1]` 当成 next-token 分数。
3. 为 n=0、空 prompt、非法 ID、容量不足分别设计接口行为检查，不写训练循环。

先写出你的推理，不把本文的手算示例改名为练习答案，也不要直接把“没有报错”当作理解了生成循环。

## 固定版本依据与材料状态

- [MLX v0.32.2 ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)：`argmax` 的 axis/keepdims/uint32、`concatenate` 的 axis，以及 isfinite/all。
- [MLX v0.32.2 indexing guide](https://github.com/ml-explore/mlx/blob/v0.32.2/docs/src/usage/indexing.rst)：整数索引、数组索引与新增轴的语义。
- [MLX v0.32.2 transforms.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/transforms.cpp)：显式求值接口。

本次只改写教学组织并做静态 JSON、语法和函数结构对照；四个原 helper/生成函数的计算与检查逻辑保持不变。代码单元全部 `execution_count=null`、`outputs=[]`，未导入 MLX、未执行本文示例或用户工作副本。纸面结果不等于实测结果，材料发布不等于学员掌握。